# 04 — Benchmark so sánh phương pháp

Phần đầu chạy benchmark nhanh phục vụ trình bày. Phần cuối là benchmark chính
thức gồm 60 lượt chạy; mặc định tắt để tránh vô tình tốn nhiều thời gian.
Checkpoint được đồng bộ sang Drive nên có thể tiếp tục sau khi Colab ngắt phiên.

In [ ]:
#@title Clone dự án từ GitHub và cài môi trường Colab
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
from pathlib import Path
import shutil
import subprocess
import sys

THU_MUC_COLAB_DRIVE = Path("/content/drive/MyDrive/Genetic_ALO_Colab")
THU_MUC_COLAB_DRIVE.mkdir(parents=True, exist_ok=True)
THU_MUC_DU_AN = Path("/content/genetic-alo")
THU_MUC_KET_QUA_DRIVE = THU_MUC_COLAB_DRIVE / "latest_outputs"
REPOSITORY_URL = "https://github.com/duktrung05/genetic-alo.git"

shutil.rmtree(THU_MUC_DU_AN, ignore_errors=True)
subprocess.run(
    [
        "git", "clone", "--depth", "1", "--branch", "main",
        REPOSITORY_URL, str(THU_MUC_DU_AN),
    ],
    check=True,
)

# Khôi phục output mới nhất từ Drive nếu notebook trước đã tạo kết quả.
if THU_MUC_KET_QUA_DRIVE.is_dir():
    shutil.copytree(
        THU_MUC_KET_QUA_DRIVE,
        THU_MUC_DU_AN / "outputs",
        dirs_exist_ok=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "-r",
        str(THU_MUC_DU_AN / "requirements.txt"),
    ],
    check=True,
)

os.chdir(THU_MUC_DU_AN)
if str(THU_MUC_DU_AN) not in sys.path:
    sys.path.insert(0, str(THU_MUC_DU_AN))

def dong_bo_ket_qua() -> Path:
    """Copy all current outputs to Drive so another notebook can reuse them."""
    THU_MUC_KET_QUA_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copytree(
        THU_MUC_DU_AN / "outputs",
        THU_MUC_KET_QUA_DRIVE,
        dirs_exist_ok=True,
    )
    return THU_MUC_KET_QUA_DRIVE

print(f"✅ Đã clone dự án tại: {THU_MUC_DU_AN}")
print(f"✅ Python: {sys.version.split()[0]}")
subprocess.run(["git", "log", "-1", "--oneline"], cwd=THU_MUC_DU_AN, check=True)
print("✅ Dataset Excel nằm trong data/instances.")

In [ ]:
#@title Benchmark nhanh
CHAY_BENCHMARK_NHANH = True #@param {type:"boolean"}
CAC_PHUONG_PHAP = "ga_repair_sls,ga_repair,ga" #@param {type:"string"}
CAC_SEED = "0-2" #@param {type:"string"}
TEN_DATASET = "easy" #@param ["easy", "medium"]

if CHAY_BENCHMARK_NHANH:
    subprocess.run(
        [
            sys.executable, "main_benchmark.py",
            "--mode", "fast",
            "--methods", CAC_PHUONG_PHAP,
            "--seeds", CAC_SEED,
            "--data-source", "excel",
            "--input", str(THU_MUC_DU_AN / f"data/instances/instance_{TEN_DATASET}.xlsx"),
            "--experiment-name", "colab_fast",
        ],
        cwd=THU_MUC_DU_AN,
        check=True,
    )
    print(f"✅ Đã đồng bộ sang: {dong_bo_ket_qua()}")
else:
    print("ℹ️ Đã bỏ qua benchmark nhanh.")

In [ ]:
#@title Xem kết quả benchmark nhanh
import pandas as pd
from IPython.display import display

thu_muc_benchmark_nhanh = THU_MUC_DU_AN / "outputs/benchmark/colab_fast"
tep_summary = thu_muc_benchmark_nhanh / "summary.csv"
if tep_summary.is_file():
    display(pd.read_csv(tep_summary))
else:
    print("Kết quả được tạo trong:", thu_muc_benchmark_nhanh)
    print([str(path.relative_to(THU_MUC_DU_AN)) for path in thu_muc_benchmark_nhanh.glob("*")])

In [ ]:
#@title Benchmark chính thức 60 lượt chạy (tùy chọn)
CHAY_BENCHMARK_CUOI = False #@param {type:"boolean"}
CHAY_LAI_TU_DAU = False #@param {type:"boolean"}

if CHAY_BENCHMARK_CUOI:
    lenh = [sys.executable, "scripts/run_final_benchmark.py"]
    if CHAY_LAI_TU_DAU:
        lenh.append("--fresh")
    subprocess.run(lenh, cwd=THU_MUC_DU_AN, check=True)
    print(f"✅ Benchmark hoàn tất; đã đồng bộ sang: {dong_bo_ket_qua()}")
else:
    print("ℹ️ Benchmark 60 lượt đang tắt. Chỉ bật khi có đủ thời gian chạy.")